In [1]:
import os                   # 운영체제 관련 표준 라이브러리
import pathlib              # 파일 경로를 객체로 다루는 표준 라이브러리

here = pathlib.Path.cwd()   # 현재 작업 위치 확인

ROOT = here.parents[2] if here.name == "day02" else here

os.chdir(ROOT) # 루트 디렉토리 설정 : 앞으로 모든 상대경로는 이 폴더가 기준이 된다.

## 로깅

In [ ]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(name)s : %(message)s",
    datefmt="%H:%M:%S", # 시:분:초
    force=True
)

# %(asctime)s : 시간
# %(levelname)s : 로그레벨
# %(name)s : 로거 이름
# %(message)s : 메세지

# 이름을 붙혀 로거를 하나 가져온다.
log = logging.getLogger("demo")
log.debug("디버그 모드의 debug입니다.")
log.info("인포레벨 로그입니다.")
log.warning("경고경고 주의하세요.")
log.error("에러발생~~~~~~~~")

12:48:43 DEBUG    demo : 디버그 모드의 debug입니다.
12:48:43 INFO     demo : 인포레벨 로그입니다.
12:48:43 WARNING  demo : 경고경고 주의하세요.
12:48:43 ERROR    demo : 에러발생~~~~~~~~


In [5]:
import sys, logging

if "backend" not in sys.path:
    sys.path.insert(0, "backend")

# 앞 셀에서 루트 로거를 건드렸으면, 아래와 같이 깨끗이하고 새로 얹는다.
logging.getLogger().handlers = []
for m in [k for k in list(sys.modules) if k.startswith("app.core.logging")]:
    del sys.modules[m]

#--------------------------------------
from app.core.logging import get_logger

logger = get_logger("app.services.document") # 임시 패키지명
logger.info("문서 적재 완료 : %s", "DOC-HR-001") # f"" ->%s 권장
logger.warning("임계치에 근접합니다. 주의하세요.")



14:51:15 INFO     app.services.document : 문서 적재 완료 : DOC-HR-001
14:51:15 WARNING  app.services.document : 임계치에 근접합니다. 주의하세요.


In [21]:
class AgentError(Exception):
    status_code, code = 400, "agent_error"

    def __init__(self, message: str, *, detail: str | None = None):
        super().__init__(message)  # print(e) -> 메세지 나옴
        self.message ,self.detail = message, detail


# 요청한 자원이 없다
class NotFound(AgentError):
    status_code ,code = 404, "not_found"

class ExternalServiceError(AgentError):
    status_code ,code = 502, "external_service_error"

def fake_parsing(doc_id):
    if doc_id == "DOC-BAD":
        raise ConnectionError("Connection refuesd: api.llm.ai:443")
    return f"{doc_id}의 본문 텍스트"

def ingest(doc_id):
    logger.info("적재 시작: %s", doc_id)
    try:
        text = fake_parsing(doc_id)
    except ConnectionError as e:
        logger.exception("문서 파싱 실패 : %s", doc_id)
        # 우리 타입의 예외 발생tlzla
        raise ExternalServiceError("문서 변환 서비스에 연결하지 못했습니다.", detail=str(e)) from e
    
    logger.info("적재 완료 : %s", doc_id)
    return text

In [18]:
ingest("HOC-DOC-001")

15:24:31 INFO     app.services.document : 적재 시작: HOC-DOC-001
15:24:31 INFO     app.services.document : 적재 완료 : HOC-DOC-001


'HOC-DOC-001의 본문 텍스트'

In [22]:
try:
    ingest("DOC-BAD")
except AgentError as e:
        print("Agent Error")
        print(e.status_code)
        print(e.code)
        print(e.message)

15:38:34 INFO     app.services.document : 적재 시작: DOC-BAD
15:38:34 ERROR    app.services.document : 문서 파싱 실패 : DOC-BAD
Traceback (most recent call last):
  File "/var/folders/d4/kztpfgqd0098tjpb6_64rlzh0000gn/T/ipykernel_45620/3913344529.py", line 24, in ingest
    text = fake_parsing(doc_id)
           ^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/d4/kztpfgqd0098tjpb6_64rlzh0000gn/T/ipykernel_45620/3913344529.py", line 18, in fake_parsing
    raise ConnectionError("Connection refuesd: api.llm.ai:443")
ConnectionError: Connection refuesd: api.llm.ai:443
Agent Error
502
external_service_error
문서 변환 서비스에 연결하지 못했습니다.
